# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("Dataset title:", metadata['name'])
print("Description:", metadata['description'])
print("Published at:", metadata['datePublished'])
print("Version:", metadata['version'])


## 2. Data Overview
Review available record sets, fields, and their IDs.


In [ ]:
# Explore record sets and their fields, referencing all entities by their `@id`.
record_sets = list(dataset.record_sets)
print(f"Available record sets ({len(record_sets)}):")
for rs in record_sets:
    rs_id = rs['@id']
    print(f"  RecordSet @id: {rs_id}\n    name: {rs.get('name')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("    Fields and their @id:")
    for f in fields:
        if isinstance(f, dict):
            print(f"      - {f.get('@id')} (name: {f.get('name')}, dataType: {f.get('dataType')})")
        else:
            print(f"      - {f}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.


In [ ]:
# Extract data from all record sets defined by their `@id`.
# Use the output of the previous cell to identify the record set IDs.

# Get all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record Set: {record_set_id}, {len(df)} records. Columns:")
    print(df.columns.tolist())
    print()
# For demonstration, select the first record set
main_record_set_id = record_set_ids[0]
print(f"Showing head of the main record set: {main_record_set_id}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.


In [ ]:
# Select a numeric field (by its @id) for analysis.
# Use field @ids from the record set inspected above.
df = dataframes[main_record_set_id]

# Example: suppose the main survey record set has GAD-7 total score with field @id 'gad7_total_score'
# You may need to adjust this to your actual field ids as shown in section 2 output, e.g. 'gad7_total_score' or similar
possible_numeric_fields = [col for col in df.columns if 'score' in col or df[col].dtype in ['int64', 'float64']]
print("Detected possible numeric fields:", possible_numeric_fields)

# For demonstration, select first available numeric field
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    # Fallback example
    numeric_field_id = df.columns[0]

threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype in ['int64', 'float64'] else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Find a grouping field for demonstration (e.g., 'gender' or 'village')
group_fields = [col for col in df.columns if ('gender' in col.lower() or 'village' in col.lower() or 'age_group' in col.lower())]

if group_fields:
    group_field = group_fields[0]
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field} (showing means):")
    print(grouped_df.head())
else:
    print("No group field like 'gender', 'village', or 'age_group' was found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if possible_numeric_fields:
    # Histogram for the selected numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    
    # Boxplot by group, if group_field exists
    if group_fields:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()
else:
    print("No numeric fields to plot.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The `mlcroissant` library simplifies the process of loading and interacting with Croissant-structured datasets.
- Record sets, fields, and values can conveniently be referenced by their `@id`, ensuring a consistent, schema-based approach for dataset exploration and downstream analysis.
- After basic inspection and EDA, you can continue with more advanced analyses tailored to your research questions.
